In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd

from lightgbm import LGBMRegressor
from scipy.stats import pearsonr


def train(X_train, y_train, model_directory_path, loop_moon, embargo):

    feature_cols = [
        c for c in X_train.columns
        if c.startswith("Feature_")
    ]

    # ============================================================
    # Merge target
    # ============================================================

    df = X_train.merge(
        y_train[["moon", "id", "target"]],
        on=["moon", "id"]
    )

    df = df.dropna(subset=["target"])

    all_moons = sorted(df["moon"].unique())

    # ============================================================
    # OPTION A:
    # Honest time-aware validation split
    # ============================================================

    cutoff = int(len(all_moons) * 0.80)

    train_moons = all_moons[:cutoff]
    val_moons   = all_moons[cutoff:]

    df_tr = df[df["moon"].isin(train_moons)].copy()
    df_val = df[df["moon"].isin(val_moons)].copy()

    print("Time-aware validation split")
    print(f"  Train moons : {len(train_moons)}")
    print(f"  Val moons   : {len(val_moons)}")

    # ============================================================
    # OPTION C:
    # Exponential recency weighting
    # ============================================================

    # More recent moons receive larger weights
    # Weight range roughly:
    # oldest moon ≈ 0.1
    # newest moon ≈ 1.0

    moon_to_rank = {
        moon: i
        for i, moon in enumerate(sorted(train_moons))
    }

    moon_ranks = df_tr["moon"].map(moon_to_rank).values

    normalized_rank = moon_ranks / moon_ranks.max()

    # EXPONENTIAL RECENCY WEIGHTS
    # Tunable parameter:
    RECENCY_STRENGTH = 8.0

    sample_weights = np.exp(
        normalized_rank * RECENCY_STRENGTH
    )

    print("\nRecency weighting")
    print(f"  Min weight : {sample_weights.min():.4f}")
    print(f"  Max weight : {sample_weights.max():.4f}")

    # ============================================================
    # Train validation model
    # ============================================================

    model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    model.fit(
        df_tr[feature_cols].fillna(0),
        df_tr["target"],
        sample_weight=sample_weights
    )

    # ============================================================
    # Validation IC
    # ============================================================

    val_preds = model.predict(
        df_val[feature_cols].fillna(0)
    )

    df_val = df_val.copy()
    df_val["prediction"] = val_preds

    corrs = []

    for moon in val_moons:

        grp = df_val[df_val["moon"] == moon]

        if len(grp) < 2:
            continue

        r, _ = pearsonr(
            grp["prediction"],
            grp["target"]
        )

        if not np.isnan(r):
            corrs.append(r)

    mean_ic = np.mean(corrs)

    print(f"\nValidation Mean IC = {mean_ic:.5f}")

    # ============================================================
    # FINAL PRODUCTION RETRAIN
    # OPTION A:
    # Retrain on ALL moons after validation
    # ============================================================

    print("\nRetraining on ALL moons...")

    # Recency weights again — now for ALL data
    full_moon_to_rank = {
        moon: i
        for i, moon in enumerate(sorted(all_moons))
    }

    full_ranks = df["moon"].map(full_moon_to_rank).values

    full_normalized = full_ranks / full_ranks.max()

    full_weights = np.exp(
        full_normalized * RECENCY_STRENGTH
    )

    final_model = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

    final_model.fit(
        df[feature_cols].fillna(0),
        df["target"],
        sample_weight=full_weights
    )

    # ============================================================
    # Save
    # ============================================================

    joblib.dump(
        {
            "model": final_model,
            "features": feature_cols,
            "validation_ic": mean_ic,
            "recency_strength": RECENCY_STRENGTH,
        },
        os.path.join(model_directory_path, "model.joblib")
    )

    print("\nModel saved.")

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):

    obj = joblib.load(
        os.path.join(model_directory_path, "model.joblib")
    )

    model = obj["model"]
    features = obj["features"]

    preds = model.predict(
        X_test[features].fillna(0)
    )

    return pd.DataFrame({
        "moon": X_test["moon"].values,
        "id": X_test["id"].values,
        "prediction": preds,
    })

In [ ]:
import os
import numpy as np
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"

os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []

for moon in splits["reduced_cloud"]:
    pred = infer(
        X_test_cloud[X_test_cloud["moon"] == moon],
        model_dir,
        loop_moon=moon,
        embargo=embargo
    )
    results.append(pred)

predictions = pd.concat(results)

corrs = []

for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon],
        on=["moon", "id"]
    )

    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")

print(f"\nMean IC = {np.mean(corrs):.4f}")